In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import datetime
from datetime import date, timedelta
import requests
from google.colab import userdata
import time
from tqdm import tqdm

In [ ]:
FINBERT_MODEL = None
FINBERT_TOKENIZER = None

def load_finbert_model():
    global FINBERT_MODEL, FINBERT_TOKENIZER
    if FINBERT_MODEL is None:
        try:
            FINBERT_TOKENIZER = AutoTokenizer.from_pretrained("ProsusAI/finbert")
            FINBERT_MODEL = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
            FINBERT_MODEL.eval()
            print("FinBERT model loaded successfully.")
        except Exception as e:
            print("Error loading FinBERT model: {}".format(e))
            print("Please ensure you have 'torch' and 'transformers' installed.")
            FINBERT_MODEL = None
    return FINBERT_MODEL, FINBERT_TOKENIZER

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_finbert_sentiment(text_series, batch_size=32):
    model, tokenizer = load_finbert_model()

    if model is None or tokenizer is None:
        print("Model not loaded. Returning zeros.")
        return [0.0] * len(text_series)

    model.to(device)

    all_scores = []

    texts = text_series.tolist()

    for i in tqdm(range(0, len(texts), batch_size), desc="FinBERT Batches"):
        batch_texts = texts[i:i + batch_size]

        batch_texts = [str(text) if pd.notna(text) else "" for text in batch_texts]

        try:
            tokens = tokenizer(
                batch_texts,
                return_tensors='pt',
                max_length=512,
                truncation=True,
                padding=True
            )

            tokens = {key: val.to(device) for key, val in tokens.items()}

            with torch.no_grad():
                outputs = model(**tokens)

            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

            for pred in predictions:
                prob_positive = pred[0].item()
                prob_negative = pred[1].item()
                all_scores.append(prob_positive - prob_negative)

        except Exception as e:
            print(f"Error processing batch starting at index {i}: {e}")
            all_scores.extend([0.0] * len(batch_texts))

    return all_scores

In [ ]:
def run_sentiment_comparison(date='2026-03-25'):
    df_raw = pd.read_csv("VZ Sentiment Data Removed Duplicates.csv")

    print("--- Applying FinBERT (Deep Learning) ---")
    df_raw['FinBERT_Score (-1, 1)'] = get_finbert_sentiment(df_raw['title'], batch_size=32)

    print("\n--- Aggregating Sentiment Scores by Day ---")

    df_daily = df_raw.groupby('date').agg({
        'FinBERT_Score (-1, 1)': 'mean',
        'title': 'count'
    }).reset_index()

    df_daily.rename(columns={'title': 'Article_Count'}, inplace=True)

    df_daily['date'] = pd.to_datetime(df_daily['date'])
    df_daily = df_daily.sort_values(by='date').reset_index(drop=True)
    df_daily['date'] = df_daily['date'].dt.strftime('%Y-%m-%d')

    output_filename = 'daily_sentiment_comparison_results.csv'
    df_daily.to_csv(output_filename, index=False)
    print("Results successfully saved to {}".format(output_filename))

    print("\n--- DAILY COMPARISON RESULTS ({}) ---".format(len(df_daily)))
    print("Sentiment Analysis for GOOGL News for 28 days ending {}".format(date))
    print("\nNote: All scores are on a scale of -1 (Negative) to +1 (Positive).")

    print(df_daily.round(4).to_markdown(index=False))

    global FINBERT_MODEL, FINBERT_TOKENIZER
    del FINBERT_MODEL
    del FINBERT_TOKENIZER

    return df_daily, df_raw

In [ ]:
def display_articles_for_day(df_raw, target_date):
    print("\n--- Detailed Article Sentiments for {} ---".format(target_date))

    filtered_articles = df_raw[df_raw['date'] == target_date]

    if not filtered_articles.empty:
        display_cols = [
            'date',
            'title',
            'FinBERT_Score (-1, 1)'
        ]
        print(filtered_articles[display_cols].round(4).to_markdown(index=False))
    else:
        print("No articles found for {}.".format(target_date))

In [ ]:
df_daily_results, df_raw_results = run_sentiment_comparison(date='2026-03-25') #change data to current day or can change # of days look back
if df_raw_results is not None:
    display_articles_for_day(df_raw_results, '2026-03-25')

--- Applying FinBERT (Deep Learning) ---


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT model loaded successfully.



FinBERT Batches: 100%|██████████| 1035/1035 [01:15<00:00, 13.64it/s]



--- Aggregating Sentiment Scores by Day ---
Results successfully saved to daily_sentiment_comparison_results.csv

--- DAILY COMPARISON RESULTS (2246) ---
Sentiment Analysis for GOOGL News for 28 days ending 2026-03-25

Note: All scores are on a scale of -1 (Negative) to +1 (Positive).
| date       |   FinBERT_Score (-1, 1) |   Article_Count |
|:-----------|------------------------:|----------------:|
| 2020-01-01 |                 -0.0202 |              15 |
| 2020-01-02 |                  0.0162 |              18 |
| 2020-01-03 |                  0.0116 |              11 |
| 2020-01-04 |                 -0.5266 |               9 |
| 2020-01-05 |                  0.0307 |               5 |
| 2020-01-06 |                 -0.0105 |               8 |
| 2020-01-07 |                 -0.0168 |              13 |
| 2020-01-08 |                  0.0789 |              17 |
| 2020-01-09 |                 -0.0961 |              56 |
| 2020-01-10 |                  0.0228 |              31 |
| 202

In [ ]:
df_daily_results.shape

(2246, 3)